In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eval.cv_metrics import (
    load_predictions, per_group_metrics, aggregate_metrics, pooled_metrics,
    mean_roc_curve, wilcoxon_seed_stability, wilcoxon_class_separation,
    wilcoxon_compare_models, list_available_models,
)

MODEL_NAME = "clam_sb"
df = load_predictions(MODEL_NAME)
HAS_TEST = "test" in df["split"].values
print(f"Prédictions chargées : {{df.shape[0]}} lignes | splits : {{df['split'].unique().tolist()}}")
df.head()

In [ ]:
# ── Métriques par (seed, fold, split) ──────────────────────────────────────
# AUC et accuracy pour chaque combinaison seed × fold × split.
metrics = per_group_metrics(df)
metrics

In [ ]:
# Moyenne ± std par seed — recommandé pour k-fold
agg = aggregate_metrics(metrics)
agg

In [ ]:
# AUC sur prédictions poolées par seed — recommandé pour LOO
# (un seul point par fold rend l'AUC par-fold indéfinie)
pooled = pooled_metrics(df)
pooled

In [ ]:
# ── Courbes ROC moyennes ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))

for split, color in [("val", "tab:blue"), ("test", "tab:orange")]:
    sub = df[df["split"] == split]
    if sub.empty:
        continue
    mean_fpr, mean_tpr, std_tpr = mean_roc_curve(sub)
    mean_auc = metrics.loc[metrics["split"] == split, "auc"].mean()
    ax.plot(mean_fpr, mean_tpr, label=f"{{split}} (AUC={{mean_auc:.3f}})", color=color)
    ax.fill_between(
        mean_fpr,
        np.clip(mean_tpr - std_tpr, 0, 1),
        np.clip(mean_tpr + std_tpr, 0, 1),
        color=color, alpha=0.2,
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title(f"{{MODEL_NAME}} — ROC moyenne (toutes seeds/folds)")
ax.legend()
plt.show()

In [ ]:
# ── Détection de surapprentissage : AUC val vs test par seed ───────────────
# Le split VAL est utilisé pour sélectionner le meilleur modèle (save_best_per_seed)
# → ses métriques sont biaisées à la hausse. La différence val - test révèle l'overfit.
# Pour la comparaison finale entre architectures, toujours préférer le split TEST.
if HAS_TEST:
    val_agg  = metrics[metrics["split"] == "val"].groupby("seed")["auc"].mean().rename("auc_val")
    test_agg = metrics[metrics["split"] == "test"].groupby("seed")["auc"].mean().rename("auc_test")
    overfit_df = pd.concat([val_agg, test_agg], axis=1).dropna()
    overfit_df["gap (val-test)"] = overfit_df["auc_val"] - overfit_df["auc_test"]
    print(overfit_df.round(4).to_string())
    gap = overfit_df["gap (val-test)"].mean()
    print(f"\nGap moyen val−test : {{gap:+.4f}}")
    if gap > 0.05:
        print("⚠  Gap > 0.05 : possible surapprentissage sur le fold de validation.")
    else:
        print("✓  Gap faible : pas de surapprentissage apparent.")
else:
    print("Pas de split test disponible — comparaison val/test impossible.")

In [ ]:
# ── Sauvegarde dans le CSV global (toutes architectures) ───────────────────
OUT_DIR = Path("eval_outputs") / "cv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_PREDICTIONS_CSV = OUT_DIR / "all_models_predictions.csv"

if ALL_PREDICTIONS_CSV.exists():
    all_preds = pd.read_csv(ALL_PREDICTIONS_CSV)
    all_preds = all_preds[all_preds["model"] != MODEL_NAME]
    all_preds = pd.concat([all_preds, df], ignore_index=True)
else:
    all_preds = df.copy()

all_preds.to_csv(ALL_PREDICTIONS_CSV, index=False)
print(f"Saved {{len(df)}} rows for '{{MODEL_NAME}}' -> {{ALL_PREDICTIONS_CSV}} (total {{len(all_preds)}} rows)")

In [ ]:
# ── Wilcoxon : stabilité inter-seeds ───────────────────────────────────────
# Split utilisé : VAL
# Justification : le val fournit k paires (une par fold) par couple de seeds,
# ce qui donne plus de puissance statistique que le test (où un seul AUC poolé
# est disponible par seed). Le biais optimiste du val est assumé ici car on
# cherche à détecter une instabilité d'entraînement, pas à estimer la perf absolue.
seeds = sorted(metrics["seed"].unique())
val_metrics = metrics[metrics["split"] == "val"]

if len(seeds) >= 2:
    stat, p, n = wilcoxon_seed_stability(val_metrics, seeds[0], seeds[-1])
    print(f"Wilcoxon signé (AUC par fold) — seed {{seeds[0]}} vs seed {{seeds[-1]}}")
    print(f"  stat={{stat}}, p={{p:.4g}}, n_paires={{n}}")
    if not np.isnan(p):
        print("  => différence significative entre seeds (p<0.05)" if p < 0.05
              else "  => pas de différence significative entre seeds")
else:
    print("Une seule seed disponible — pas de test de stabilité entre seeds.")

In [ ]:
# ── Wilcoxon : séparation des classes ──────────────────────────────────────
# Split utilisé : TEST (non biaisé par la sélection de modèle)
# Fallback sur VAL si test absent.
# Question : le modèle attribue-t-il des probabilités significativement
# plus élevées à la classe 1 qu'à la classe 0 ?
primary = "test" if HAS_TEST else "val"
stat, p = wilcoxon_class_separation(df, split=primary)
print(f"Wilcoxon rank-sum (proba classe 1 vs classe 0) — split={primary}")
print(f"  stat={{stat:.4f}}, p={{p:.4g}}")
print("  => séparation significative (p<0.05)" if p < 0.05
      else "  => séparation non significative")

In [ ]:
# ── Wilcoxon : comparaison inter-architectures ─────────────────────────────
# Split utilisé : TEST — comparaison finale non biaisée entre architectures.
# Fallback sur VAL si test absent.
# Les paires sont appariées par (seed, fold) pour le test signé de Wilcoxon.
primary = "test" if HAS_TEST else "val"
other_models = [m for m in list_available_models() if m != MODEL_NAME]
print(f"Modèles disponibles pour comparaison : {{other_models}}")
print(f"Split utilisé : {{primary}}\n")

comparison_rows = []
for other in other_models:
    try:
        other_df = load_predictions(other)
    except FileNotFoundError:
        continue
    other_metrics = per_group_metrics(other_df)
    a = metrics[metrics["split"] == primary]
    b = other_metrics[other_metrics["split"] == primary]
    if a.empty or b.empty:
        continue
    stat, p, n = wilcoxon_compare_models(a, b, metric="auc")
    comparison_rows.append({
        "model_a": MODEL_NAME, "model_b": other, "split": primary,
        "auc_a": a["auc"].mean(), "auc_b": b["auc"].mean(),
        "wilcoxon_stat": stat, "p_value": p, "n_pairs": n,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df